In [ ]:
import logging
from importlib import reload
import os

import torch
import torch.nn as nn
import cv2
import numpy as np
from tqdm import trange

from pathlib import Path

from marmopose.version import __version__ as marmopose_version
from marmopose.config import Config
from marmopose.processing.prediction import Predictor
import matplotlib.pyplot as plt
import matplotlib.patches as patches

import json


logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(name)s - %(message)s')
logger = logging.getLogger(__name__)

logger.info(f'MarmoPose version: {marmopose_version}')

In [ ]:
n_training_examples = []
for directory in os.listdir('../data'):
    if directory.startswith('pose_model_finetune_'):
        n_training_examples.append(int(directory[20:]))
print(n_training_examples)

In [ ]:
config_path = '../configs/default.yaml'

config = Config(
    config_path=config_path,
    
    n_tracks=1,
    project='../demos/test',
    det_model= '../data/detection_model_finetune',
    pose_model= '../data/pose_model_finetune',

)
print(config.sub_directory)


config_base = Config(
    config_path=config_path,
    
    n_tracks=1,
    project='../demos/test',
)
print(config.sub_directory)

In [ ]:
predictor = Predictor(config, batch_size=4)
predictor_base = Predictor(config_base, batch_size=4)

In [ ]:
import json
dataset_dir = '../../Sleap/TestData3D/marmoset_family/'
with open(os.path.join(dataset_dir,'annotations/all.json'), 'r') as f:
    test_json = json.load(f)
img_ids = [ann['image_id'] for ann in test_json['annotations']]
images = [cv2.imread(os.path.join(dataset_dir,'images',img['file_name'])) for img in test_json['images'] if img['id'] in img_ids]
gt_keypoints = np.array([ann['keypoints'] for ann in test_json['annotations']]).reshape((-1,1,16,3))
gt_bboxes = np.array([ann['bbox'] for ann in test_json['annotations']]).reshape((-1,1,4))


In [ ]:
sorted_indices = np.load('../../Sleap/TestData3D/sorted_indices.npy')
missing_indices = np.load('../../Sleap/TestData3D/missing_indices.npy')
sorted_images = np.array(images)[sorted_indices]
sorted_gt_keypoints = gt_keypoints[sorted_indices]
sorted_gt_bboxes = gt_bboxes[sorted_indices]


In [ ]:
import gc
torch.cuda.empty_cache()
gc.collect()
points_with_score_2d_finetuned_part1, bboxes_finetuned_part1 = predictor.predict_image_batch(sorted_images[:100])
torch.cuda.empty_cache()
points_with_score_2d_finetuned_part2, bboxes_finetuned_part2 = predictor.predict_image_batch(sorted_images[100:])
torch.cuda.empty_cache()
points_with_score_2d_base_part1, bboxes_base_part1 = predictor_base.predict_image_batch(sorted_images[:100])
torch.cuda.empty_cache()
points_with_score_2d_base_part2, bboxes_base_part2 = predictor_base.predict_image_batch(sorted_images[100:])
torch.cuda.empty_cache()
points_with_score_2d_finetuned = np.concatenate((points_with_score_2d_finetuned_part1,points_with_score_2d_finetuned_part2), axis=0)
bboxes_finetuned = np.concatenate((bboxes_finetuned_part1,bboxes_finetuned_part2), axis=0)
points_with_score_2d_base = np.concatenate((points_with_score_2d_base_part1,points_with_score_2d_base_part2), axis=0)
bboxes_base = np.concatenate((bboxes_base_part1,bboxes_base_part2), axis=0)
points_with_score_2d_base[:,:,:,2] /= np.nanmax(points_with_score_2d_base[:,:,:,2])
points_with_score_2d_finetuned[:,:,:,2] /= np.nanmax(points_with_score_2d_finetuned[:,:,:,2])


In [ ]:
configs_dict = {}
predictor_dict = {}
points_with_score_2d_dict = {}
bbox_dict = {}
for n in n_training_examples:
    configs_dict[n] = Config(
        config_path=config_path,
        n_tracks=1,
        project='../demos/test',
        det_model= '../data/detection_model_finetune',
        pose_model= f'../data/pose_model_finetune_{n}',
        )
    predictor_dict[n] = Predictor(configs_dict[n], batch_size=4)

    torch.cuda.empty_cache()
    gc.collect()
    points_part1, bboxes_part1 = predictor_dict[n].predict_image_batch(sorted_images[:100])
    torch.cuda.empty_cache()
    points_part2, bboxes_part2 = predictor_dict[n].predict_image_batch(sorted_images[100:])
    torch.cuda.empty_cache()
    points_with_score_2d_dict[n] = np.concatenate((points_part1,points_part2), axis=0)
    points_with_score_2d_dict[n][:,:,:,2] /= np.nanmax(points_with_score_2d_dict[n][:,:,:,2])
    bbox_dict[n] = np.concatenate((bboxes_part1,bboxes_part2), axis=0)


In [ ]:
def compute_perc_correct_per_threshold_2D(groundtruth, predicted, thresholds):
    non_labelled_head_gt = np.sum(groundtruth[:,:,:3,2] == 0)
    labelled_head_gt = groundtruth[:,:,:3,2].size - non_labelled_head_gt
    non_labelled_body_gt = np.sum(groundtruth[:,:,[3,8],2] == 0)
    labelled_body_gt = groundtruth[:,:,[3,8],2].size - non_labelled_body_gt
    non_labelled_limbs_gt = np.sum(groundtruth[:,:,np.r_[4:8,9:13],2] == 0)
    labelled_limbs_gt = groundtruth[:,:,np.r_[4:8,9:13],2].size - non_labelled_limbs_gt
    non_labelled_tail_gt = np.sum(groundtruth[:,:,13:,2] == 0)
    labelled_tail_gt = groundtruth[:,:,13:,2].size - non_labelled_tail_gt

    error_head_finetuned = (predicted[:,:,:3,:2] - groundtruth[:,:,:3,:2])
    error_head_finetuned = np.sqrt(np.einsum('ijkl, ijkl -> ik',error_head_finetuned,error_head_finetuned))
    perc_head_finetuned = [100 * np.sum(error_head_finetuned < thresh)/labelled_head_gt for thresh in thresholds]
    error_body_finetuned = (predicted[:,:,[3,8],:2] - groundtruth[:,:,[3,8],:2])
    error_body_finetuned = np.sqrt(np.einsum('ijkl, ijkl -> ik',error_body_finetuned,error_body_finetuned))
    perc_body_finetuned = [100 * np.sum(error_body_finetuned < thresh)/labelled_body_gt for thresh in thresholds]

    error_limbs_finetuned = (predicted[:,:,np.r_[4:8,9:13],:2] - groundtruth[:,:,np.r_[4:8,9:13],:2])
    error_limbs_finetuned = np.sqrt(np.einsum('ijkl, ijkl -> ik',error_limbs_finetuned,error_limbs_finetuned))
    perc_limbs_finetuned = [100 * np.sum(error_limbs_finetuned < thresh)/labelled_limbs_gt for thresh in thresholds]

    error_tail_finetuned = (predicted[:,:,13:,:2] - groundtruth[:,:,13:,:2])
    error_tail_finetuned = np.sqrt(np.einsum('ijkl, ijkl -> ik',error_tail_finetuned,error_tail_finetuned))
    perc_tail_finetuned = [100 * np.sum(error_tail_finetuned < thresh)/labelled_tail_gt for thresh in thresholds]

    return perc_head_finetuned, perc_body_finetuned, perc_limbs_finetuned, perc_tail_finetuned

thresholds = np.arange(0,70,2)

perc_head_finetuned, perc_body_finetuned, perc_limbs_finetuned, perc_tail_finetuned = compute_perc_correct_per_threshold_2D(sorted_gt_keypoints, points_with_score_2d_finetuned, thresholds)
perc_head_base, perc_body_base, perc_limbs_base, perc_tail_base = compute_perc_correct_per_threshold_2D(sorted_gt_keypoints, points_with_score_2d_base, thresholds)
plt.plot(thresholds,perc_head_finetuned,c='b',marker = 'o',ms=4)
plt.plot(thresholds,perc_body_finetuned,c='b',marker = 's',ms=4)
plt.plot(thresholds,perc_limbs_finetuned,c='b',marker = '^',ms=4)
plt.plot(thresholds,perc_tail_finetuned,c='b',marker = 'd',ms=4)

plt.plot(thresholds,perc_head_base,c='r',marker = 'o',ms=4)
plt.plot(thresholds,perc_body_base,c='r',marker = 's',ms=4)
plt.plot(thresholds,perc_limbs_base,c='r',marker = '^',ms=4)
plt.plot(thresholds,perc_tail_base,c='r',marker = 'd',ms=4)

n_ns = len(points_with_score_2d_dict.keys())
for i, n in enumerate(sorted(points_with_score_2d_dict.keys())):
    perc_head, perc_body, perc_limbs, perc_tail = compute_perc_correct_per_threshold_2D(sorted_gt_keypoints, points_with_score_2d_dict[n], thresholds)
    plt.plot(thresholds,perc_head,c=((n_ns - i)/(n_ns + 1),(n_ns - i)/(n_ns + 1),1),marker = 'o',ms=4)
    plt.plot(thresholds,perc_body,c=((n_ns - i)/(n_ns + 1),(n_ns - i)/(n_ns + 1),1),marker = 's',ms=4)
    plt.plot(thresholds,perc_limbs,c=((n_ns - i)/(n_ns + 1),(n_ns - i)/(n_ns + 1),1),marker = '^',ms=4)
    plt.plot(thresholds,perc_tail,c=((n_ns - i)/(n_ns + 1),(n_ns - i)/(n_ns + 1),1),marker = 'd',ms=4)



plt.xlabel('Error threshold (pixels)')
plt.ylabel('Accuracy (%)')
plt.ylim((0,100))
plt.show()

In [ ]:
print(points_with_score_2d_finetuned.shape)
print(bboxes_finetuned.shape)
print(points_with_score_2d_base.shape)
print(bboxes_base.shape)
print(gt_keypoints.shape)
print(gt_bboxes.shape)
print(np.nansum((sorted_gt_bboxes - bboxes_base)**2)/(sorted_gt_keypoints.shape[0] * sorted_gt_keypoints.shape[2]))
print(np.nansum((sorted_gt_bboxes - bboxes_finetuned)**2)/(sorted_gt_keypoints.shape[0] * sorted_gt_keypoints.shape[2]))
for i, image in enumerate(sorted_images[:20]):
    fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(20, 10))
    axes[0].set_title(img_ids[i])
    axes[0].imshow(image[:,:,[2,1,0]])
    axes[1].imshow(image[:,:,[2,1,0]])
    rect = patches.Rectangle(
        (sorted_gt_bboxes[i,0,0], sorted_gt_bboxes[i,0,1]), sorted_gt_bboxes[i,0,2], sorted_gt_bboxes[i,0,3],
        linewidth=1, edgecolor='g', facecolor='none'
    )
    axes[0].add_patch(rect)
    axes[0].scatter(sorted_gt_keypoints[i,0,:,0],sorted_gt_keypoints[i,0,:,1],alpha = sorted_gt_keypoints[i,0,:,2]/2,c='g',s = 1)
    base_nans = np.isnan(points_with_score_2d_base[i,0,:,2])
    if not np.sum(base_nans) == 16:
        axes[0].scatter(points_with_score_2d_base[i,0,~base_nans,0],points_with_score_2d_base[i,0,~base_nans,1],c='r',s = points_with_score_2d_base[i,0,~base_nans,2])
    rect = patches.Rectangle(
        (sorted_gt_bboxes[i,0,0], sorted_gt_bboxes[i,0,1]), sorted_gt_bboxes[i,0,2], sorted_gt_bboxes[i,0,3],
        linewidth=1, edgecolor='g', facecolor='none'
    )
    
    axes[1].add_patch(rect)
    axes[1].scatter(sorted_gt_keypoints[i,0,:,0],sorted_gt_keypoints[i,0,:,1],alpha = sorted_gt_keypoints[i,0,:,2]/2,c='g',s = 1)
    finetuned_nans = np.isnan(points_with_score_2d_finetuned[i,0,:,2])
    axes[1].scatter(points_with_score_2d_finetuned[i,0,~finetuned_nans,0],points_with_score_2d_finetuned[i,0,~finetuned_nans,1],c='b',s = points_with_score_2d_finetuned[i,0,~finetuned_nans,2])
    rect = patches.Rectangle(
        (bboxes_base[i,0,0], bboxes_base[i,0,1]), bboxes_base[i,0,2]-bboxes_base[i,0,0], bboxes_base[i,0,3]-bboxes_base[i,0,1],
        linewidth=1, edgecolor='r', facecolor='none'
    )
    axes[0].add_patch(rect)
    rect = patches.Rectangle(
        (bboxes_finetuned[i,0,0], bboxes_finetuned[i,0,1]), bboxes_finetuned[i,0,2]-bboxes_finetuned[i,0,0], bboxes_finetuned[i,0,3]-bboxes_finetuned[i,0,1],
        linewidth=1, edgecolor='b', facecolor='none'
    )
    axes[1].add_patch(rect)

    axes[0].axis('off')
    axes[1].axis('off')
    fig.show()




In [ ]:
start_indices = np.concatenate((np.array([0]), 1 + missing_indices))
end_indices = np.concatenate((missing_indices,np.array([sorted_indices.size + missing_indices.size])))

sorted_images_with_missing = np.full((sorted_indices.size + missing_indices.size, *sorted_images.shape[1:]), 255)
sorted_gt_keypoints_with_missing = np.full((sorted_indices.size + missing_indices.size, *gt_keypoints.shape[1:]), np.nan)
sorted_base_keypoints_with_missing = np.full((sorted_indices.size + missing_indices.size, *points_with_score_2d_base.shape[1:]), np.nan)
sorted_finetuned_keypoints_with_missing = np.full((sorted_indices.size + missing_indices.size, *points_with_score_2d_finetuned.shape[1:]), np.nan)
for i, (idx0, idx1) in enumerate(zip(start_indices,end_indices)):
    print(i,idx0,idx1)
    sorted_images_with_missing[idx0:idx1, ...] = sorted_images[idx0-i:idx1-i, ...]
    sorted_gt_keypoints_with_missing[idx0:idx1, ...] = sorted_gt_keypoints[idx0 -i:idx1 - i, ...]
    sorted_base_keypoints_with_missing[idx0:idx1, ...] = points_with_score_2d_base[idx0 -i:idx1 - i, ...]
    sorted_finetuned_keypoints_with_missing[idx0:idx1, ...] = points_with_score_2d_finetuned[idx0 -i:idx1 - i, ...]

sorted_images_with_missing = sorted_images_with_missing.reshape(4,-1,*sorted_images_with_missing.shape[1:])
sorted_gt_keypoints_with_missing = sorted_gt_keypoints_with_missing.reshape(4,-1,*sorted_gt_keypoints_with_missing.shape[2:])
sorted_gt_keypoints_with_missing[sorted_gt_keypoints_with_missing[:,:,:,2] == 0,:] = np.nan
print(sorted_gt_keypoints_with_missing.shape)
sorted_base_keypoints_with_missing = sorted_base_keypoints_with_missing.reshape(4,-1,*sorted_base_keypoints_with_missing.shape[2:])
sorted_finetuned_keypoints_with_missing = sorted_finetuned_keypoints_with_missing.reshape(4,-1,*sorted_finetuned_keypoints_with_missing.shape[2:])

In [ ]:
for idx in range(20):
    for i, image in enumerate(sorted_images_with_missing):
        image = image[idx]
        fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(20, 10))
        axes[0].set_title(f'{idx} {img_ids[i]}')
        axes[0].imshow(image[:,:,[2,1,0]])
        axes[1].imshow(image[:,:,[2,1,0]])
        gt_nans = np.isnan(sorted_gt_keypoints_with_missing[i,idx,:,2])
        if not np.sum(gt_nans) == 16:
            axes[0].scatter(sorted_gt_keypoints_with_missing[i,idx,~gt_nans,0],sorted_gt_keypoints_with_missing[i,idx,~gt_nans,1],alpha = sorted_gt_keypoints_with_missing[i,idx,~gt_nans,2]/2,c='g',s = 1)
            axes[1].scatter(sorted_gt_keypoints_with_missing[i,idx,~gt_nans,0],sorted_gt_keypoints_with_missing[i,idx,~gt_nans,1],alpha = sorted_gt_keypoints_with_missing[i,idx,~gt_nans,2]/2,c='g',s = 1)
        base_nans = np.isnan(sorted_base_keypoints_with_missing[i,idx,:,2])
        if not np.sum(base_nans) == 16:
            axes[0].scatter(sorted_base_keypoints_with_missing[i,idx,~base_nans,0],sorted_base_keypoints_with_missing[i,idx,~base_nans,1],c='r',s = sorted_base_keypoints_with_missing[i,idx,~base_nans,2])
        finetuned_nans = np.isnan(sorted_finetuned_keypoints_with_missing[i,idx,:,2])
        axes[1].scatter(sorted_finetuned_keypoints_with_missing[i,idx,~finetuned_nans,0],sorted_finetuned_keypoints_with_missing[i,idx,~finetuned_nans,1],c='b',s = sorted_finetuned_keypoints_with_missing[i,idx,~finetuned_nans,2])

        axes[0].axis('off')
        axes[1].axis('off')
        fig.show()




In [ ]:
from tqdm import trange
from marmopose.calibration.cameras import CameraGroup

camera_group = CameraGroup.load_from_json("../demos/single/calibration/camera_params.json")
def triangulate_frame(points_with_score_2d: np.ndarray, ransac=True):
    """
    Args:
        points_with_score_2d: (n_cams, n_tracks, n_bodyparts, (x, y, score))
    
    Returns:
        points_3d: (n_bodyparts, (x, y, z))
    """


    if ransac:
        points_3d = camera_group.triangulate_ransac(points_with_score_2d, undistort=True)
    else:
        points_3d = camera_group.triangulate(points_with_score_2d, undistort=True)
        
    return points_3d


n_cams, n_frames, n_bodyparts, n_dim = sorted_gt_keypoints_with_missing.shape
gt_points_3d = np.full((n_frames, n_bodyparts, 3), np.nan)
base_points_3d = np.full((n_frames, n_bodyparts, 3), np.nan)
finetuned_points_3d = np.full((n_frames, n_bodyparts, 3), np.nan)

for frame_idx in trange(n_frames, ncols=100, desc='Triangulating... ', unit='frames'):
    gt_all_points_with_score_2d_frame = sorted_gt_keypoints_with_missing[:, frame_idx]
    gt_point_3d = triangulate_frame(gt_all_points_with_score_2d_frame, ransac=True) 
    gt_points_3d[frame_idx] = gt_point_3d
    base_all_points_with_score_2d_frame = sorted_base_keypoints_with_missing[:, frame_idx]
    base_point_3d = triangulate_frame(base_all_points_with_score_2d_frame, ransac=True) 
    base_points_3d[frame_idx] = base_point_3d

    finetuned_all_points_with_score_2d_frame = sorted_finetuned_keypoints_with_missing[:, frame_idx]
    finetuned_point_3d = triangulate_frame(finetuned_all_points_with_score_2d_frame, ransac=True) 
    finetuned_points_3d[frame_idx] = finetuned_point_3d



In [ ]:
def compute_perc_correct_per_threshold_3D(groundtruth, predicted, thresholds):
    gt_head = groundtruth[:,:3,:]
    non_labelled_head_gt = np.sum(np.isnan(gt_head[:,:,2]))
    labelled_head_gt = gt_head[:,:,2].size - non_labelled_head_gt
    gt_body = groundtruth[:,[3,8],:]
    non_labelled_body_gt = np.sum(np.isnan(gt_body[:,:,2]))
    labelled_body_gt = gt_body[:,:,2].size - non_labelled_body_gt
    gt_limbs = groundtruth[:,np.r_[4:8,9:13],:]
    non_labelled_limbs_gt = np.sum(np.isnan(gt_limbs[:,:,2]))
    labelled_limbs_gt = gt_limbs[:,:,2].size - non_labelled_limbs_gt
    gt_tail = groundtruth[:,13:,:]
    non_labelled_tail_gt = np.sum(np.isnan(gt_tail[:,:,2]))
    labelled_tail_gt = gt_tail[:,:,2].size - non_labelled_tail_gt

    error_head_finetuned = (predicted[:,:3,:] - gt_head)
    error_head_finetuned = np.sqrt(np.einsum('ijk, ijk -> ij',error_head_finetuned,error_head_finetuned))
    perc_head_finetuned = [100 * np.sum(error_head_finetuned < thresh)/labelled_head_gt for thresh in thresholds]

    error_body_finetuned = (predicted[:,[3,8],:] - gt_body)
    error_body_finetuned = np.sqrt(np.einsum('ijk, ijk -> ij',error_body_finetuned,error_body_finetuned))
    perc_body_finetuned = [100 * np.sum(error_body_finetuned < thresh)/labelled_body_gt for thresh in thresholds]

    error_limbs_finetuned = (predicted[:,np.r_[4:8,9:13],:] - gt_limbs)
    error_limbs_finetuned = np.sqrt(np.einsum('ijk, ijk -> ij',error_limbs_finetuned,error_limbs_finetuned))
    perc_limbs_finetuned = [100 * np.sum(error_limbs_finetuned < thresh)/labelled_limbs_gt for thresh in thresholds]

    error_tail_finetuned = (predicted[:,13:,:] - gt_tail)
    error_tail_finetuned = np.sqrt(np.einsum('ijk, ijk -> ij',error_tail_finetuned,error_tail_finetuned))
    perc_tail_finetuned = [100 * np.sum(error_tail_finetuned < thresh)/labelled_tail_gt for thresh in thresholds]

    return perc_head_finetuned, perc_body_finetuned, perc_limbs_finetuned, perc_tail_finetuned


In [ ]:
thresholds = np.arange(0,100,4)

perc_head_finetuned, perc_body_finetuned, perc_limbs_finetuned, perc_tail_finetuned = compute_perc_correct_per_threshold_3D(gt_points_3d, finetuned_points_3d, thresholds)
perc_head_base, perc_body_base, perc_limbs_base, perc_tail_base = compute_perc_correct_per_threshold_3D(gt_points_3d, base_points_3d, thresholds)

plt.plot(thresholds,perc_head_finetuned,c='b',marker = 'o',ms=4)
plt.plot(thresholds,perc_body_finetuned,c='b',marker = 's',ms=4)
plt.plot(thresholds,perc_limbs_finetuned,c='b',marker = '^',ms=4)
plt.plot(thresholds,perc_tail_finetuned,c='b',marker = 'd',ms=4)

plt.plot(thresholds,perc_head_base,c='r',marker = 'o',ms=4)
plt.plot(thresholds,perc_body_base,c='r',marker = 's',ms=4)
plt.plot(thresholds,perc_limbs_base,c='r',marker = '^',ms=4)
plt.plot(thresholds,perc_tail_base,c='r',marker = 'd',ms=4)
plt.xlabel('Error threshold (mm)')
plt.ylabel('Accuracy (%)')
plt.ylim((0,100))
plt.show()

In [ ]:
xlim, ylim, zlim, _ = config.visualization['room_dimensions']

for i in range(30):
    # Create a 3D scatter plot
    fig = plt.figure(figsize=(10,5))
    ax1 = fig.add_subplot(131, projection='3d')
    ax2 = fig.add_subplot(132, projection='3d')
    ax3 = fig.add_subplot(133, projection='3d')
    for bodyparts in config.visualization['skeleton'][::-1]:
        idx_bodyparts = []
        for bodypart in bodyparts:
            idx_bodyparts.append(config.animal['bodyparts'].index(bodypart))
        ax1.plot(gt_points_3d[i,idx_bodyparts,0],gt_points_3d[i,idx_bodyparts,1],gt_points_3d[i,idx_bodyparts,2], marker = 'o', ms=3,color='g')
        ax2.plot(base_points_3d[i,idx_bodyparts,0],base_points_3d[i,idx_bodyparts,1],base_points_3d[i,idx_bodyparts,2], marker = 'o', ms=3,color='r')
        ax3.plot(finetuned_points_3d[i,idx_bodyparts,0],finetuned_points_3d[i,idx_bodyparts,1],finetuned_points_3d[i,idx_bodyparts,2], marker = 'o', ms=3,color='b')
    ax1.set_title(f'frame {i} ground truth')
    ax1.set_xlim((0,xlim))
    ax1.set_ylim((0,ylim))
    ax1.set_zlim((0,zlim))
    ax2.set_title(f'frame {i} base model')
    ax2.set_xlim((0,xlim))
    ax2.set_ylim((0,ylim))
    ax2.set_zlim((0,zlim))
    ax3.set_title(f'frame {i} finetuned model')
    ax3.set_xlim((0,xlim))
    ax3.set_ylim((0,ylim))
    ax3.set_zlim((0,zlim))

In [ ]:
from marmopose.utils.data_io import load_points_3d_h5
with open('../../Sleap/TestData3D/frames_dict.json') as f:
    frames_in_videos = json.load(f)

fullworkflow_finetuned_points3d = np.full((finetuned_points_3d.shape), np.nan)
fullworkflow_base_points3d = np.full((finetuned_points_3d.shape), np.nan)
i = 0
for video in sorted(frames_in_videos.keys()):
    points_3d = load_points_3d_h5(f"../../Videos/Test3.{int(video)}/Output/points_3d/optimized.h5")
    fullworkflow_finetuned_points3d[i:i+len(frames_in_videos[video]),:,:] = points_3d[0,frames_in_videos[video],:,:]
    points_3d = load_points_3d_h5(f"../../Videos/Test3.{int(video)}/Output_basemodel/points_3d/optimized.h5")
    fullworkflow_base_points3d[i:i+len(frames_in_videos[video]),:,:] = points_3d[0,frames_in_videos[video],:,:]
    i += len(frames_in_videos[video])


In [ ]:
thresholds = np.arange(0,70,2)
perc_head_fullworkflow_finetuned, perc_body_fullworkflow_finetuned, perc_limbs_fullworkflow_finetuned, perc_tail_fullworkflow_finetuned = compute_perc_correct_per_threshold_3D(gt_points_3d, fullworkflow_finetuned_points3d, thresholds)
perc_head_fullworkflow_base, perc_body_fullworkflow_base, perc_limbs_fullworkflow_base, perc_tail_fullworkflow_base = compute_perc_correct_per_threshold_3D(gt_points_3d, fullworkflow_base_points3d, thresholds)


plt.plot(thresholds,perc_head_fullworkflow_finetuned,c='b',marker = 'o',ms=4)
plt.plot(thresholds,perc_body_fullworkflow_finetuned,c='b',marker = 's',ms=4,)
plt.plot(thresholds,perc_limbs_fullworkflow_finetuned,c='b',marker = '^',ms=4)
plt.plot(thresholds,perc_tail_fullworkflow_finetuned,c='b',marker = 'd',ms=4)

plt.plot(thresholds,perc_head_fullworkflow_base,c='r',marker = 'o',ms=4)
plt.plot(thresholds,perc_body_fullworkflow_base,c='r',marker = 's',ms=4,)
plt.plot(thresholds,perc_limbs_fullworkflow_base,c='r',marker = '^',ms=4)
plt.plot(thresholds,perc_tail_fullworkflow_base,c='r',marker = 'd',ms=4)

plt.xlabel('Error threshold (mm)')
plt.ylabel('Accuracy (%)')
plt.ylim((0,100))
plt.show()

In [ ]:
xlim, ylim, zlim, _ = config.visualization['room_dimensions']

for i in range(30):
    # Create a 3D scatter plot
    fig = plt.figure(figsize=(10,5))
    ax1 = fig.add_subplot(131, projection='3d')
    ax2 = fig.add_subplot(132, projection='3d')
    ax3 = fig.add_subplot(133, projection='3d')
    for bodyparts in config.visualization['skeleton'][::-1]:
        idx_bodyparts = []
        for bodypart in bodyparts:
            idx_bodyparts.append(config.animal['bodyparts'].index(bodypart))
        ax1.plot(gt_points_3d[i,idx_bodyparts,0],gt_points_3d[i,idx_bodyparts,1],gt_points_3d[i,idx_bodyparts,2], marker = 'o', ms=3,color='g')
        ax2.plot(finetuned_points_3d[i,idx_bodyparts,0],finetuned_points_3d[i,idx_bodyparts,1],finetuned_points_3d[i,idx_bodyparts,2], marker = 'o', ms=3,color='b')
        ax3.plot(fullworkflow_finetuned_points3d[i,idx_bodyparts,0],fullworkflow_finetuned_points3d[i,idx_bodyparts,1],fullworkflow_finetuned_points3d[i,idx_bodyparts,2], marker = 'o', ms=3,color='indigo')
    ax1.set_title(f'frame {i} ground truth')
    ax1.set_xlim((0,xlim))
    ax1.set_ylim((0,ylim))
    ax1.set_zlim((0,zlim))
    ax2.set_title(f'frame {i} finetuned model')
    ax2.set_xlim((0,xlim))
    ax2.set_ylim((0,ylim))
    ax2.set_zlim((0,zlim))
    ax3.set_title(f'frame {i} fullworkflow finetuned model')
    ax3.set_xlim((0,xlim))
    ax3.set_ylim((0,ylim))
    ax3.set_zlim((0,zlim))